In [1]:
import os
import numpy as np
import tensorflow as tf
from ext.lab2im import layers
import nibabel as nib
from ext.neuron import models as nrn_models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

from tqdm import tqdm

Using TensorFlow backend.


In [2]:
# Set TensorFlow to use GPU
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
        tf.config.set_visible_devices(physical_devices[0], 'GPU')
        print("GPU is set for TensorFlow tasks.")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Using CPU instead.")

GPU is set for TensorFlow tasks.


2025-03-26 13:53:49.773617: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcuda.so.1
2025-03-26 13:53:49.784038: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-26 13:53:49.784108: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1561] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 4070 Ti computeCapability: 8.9
coreClock: 2.61GHz coreCount: 60 deviceMemorySize: 11.70GiB deviceMemoryBandwidth: 469.43GiB/s
2025-03-26 13:53:49.784226: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcudart.so.10.1
2025-03-26 13:53:49.785151: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcublas.so.10
2025-03-26 13:53:49.786135: I tensorflow/stream_executor/

In [2]:
def combined_loss(y_true, y_pred, alpha=0.5, smooth=1e-6):
    """
    Combined loss function: weighted sum of categorical cross-entropy and Dice loss.

    :param y_true: Ground truth labels (one-hot encoded or binary).
    :param y_pred: Predicted labels (probabilities from the model).
    :param alpha: Weight for the Dice loss (1 - alpha is the weight for categorical cross-entropy).
    :param smooth: Smoothing factor to avoid division by zero in Dice loss.
    :return: Combined loss value.
    """
    # Categorical cross-entropy loss
    cce_loss = tf.keras.losses.categorical_crossentropy(y_true, y_pred)

    # Dice loss
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f)
    dice_coeff = (2. * intersection + smooth) / (union + smooth)
    dice_loss = 1 - dice_coeff

    # Combined loss
    return alpha * dice_loss + (1 - alpha) * cce_loss

In [3]:
model_path = "models/brats_synthseg_2.0.h5"  # Replace with the actual path

# Define input parameters
input_shape = [None, None, None, 1]  # 3D input with 1 channel
labels_segmentation = np.arange(36)  # Example label list (update based on your dataset)
n_levels = 5
nb_conv_per_level = 2
conv_size = 3
unet_feat_count = 24
feat_multiplier = 2
activation = 'elu'
sigma_smoothing = 0
flip_indices = None
gradients = False

# Define the build_model function
def build_model(path_model,
                input_shape,
                labels_segmentation,
                n_levels,
                nb_conv_per_level,
                conv_size,
                unet_feat_count,
                feat_multiplier,
                activation,
                sigma_smoothing,
                flip_indices,
                gradients):
    assert os.path.isfile(path_model), "The provided model path does not exist."

    # Get the number of labels
    n_labels_seg = len(labels_segmentation)

    # Build the UNet
    net = nrn_models.unet(input_shape=input_shape,
                          nb_labels=n_labels_seg,
                          nb_levels=n_levels,
                          nb_conv_per_level=nb_conv_per_level,
                          conv_size=conv_size,
                          nb_features=unet_feat_count,
                          feat_mult=feat_multiplier,
                          activation=activation,
                          batch_norm=-1)
    net.load_weights(path_model, by_name=True, skip_mismatch=True)

    # Smooth posteriors if specified
    if sigma_smoothing > 0:
        last_tensor = net.output
        last_tensor = layers.GaussianBlur(sigma=sigma_smoothing)(last_tensor)
        net = tf.keras.models.Model(inputs=net.inputs, outputs=last_tensor)

    return net

# Load the model
model = build_model(model_path,
                    input_shape,
                    labels_segmentation,
                    n_levels,
                    nb_conv_per_level,
                    conv_size,
                    unet_feat_count,
                    feat_multiplier,
                    activation,
                    sigma_smoothing,
                    flip_indices,
                    gradients)

# Fine-tune the last two layers
for layer in model.layers[:-2]:
    layer.trainable = False  # Freeze all layers except the last two

# Compile the model with the combined loss
model.compile(optimizer=Adam(learning_rate=1e-4), 
              loss=lambda y_true, y_pred: combined_loss(y_true, y_pred, alpha=0.8), 
              metrics=['accuracy'])

2025-03-27 10:31:26.786315: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcuda.so.1
2025-03-27 10:31:26.789256: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-27 10:31:26.789316: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1561] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 4070 Ti computeCapability: 8.9
coreClock: 2.61GHz coreCount: 60 deviceMemorySize: 11.70GiB deviceMemoryBandwidth: 469.43GiB/s
2025-03-27 10:31:26.789424: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcudart.so.10.1
2025-03-27 10:31:26.790337: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcublas.so.10
2025-03-27 10:31:26.791330: I tensorflow/stream_executor/

In [4]:
model.summary()

Model: "unet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
_______________________________________________________________________________________________

In [6]:
model.input_shape

(None, None, None, None, 1)

In [26]:
class VolumeDataset():
    def __init__(self, image_dir, masks_dir, batch_size, total_num):
        self.image_dir = image_dir
        self.masks_dir = masks_dir
        self.batch_size = batch_size
        self.total_num = total_num
        self.start_num = 1 # Starting number

    def _load_and_stack_volumes(self, start_num):
        """
        Load a batch of volumes starting from the given start number.
        
        :param start_num: Starting index for the batch
        :return: Stacked numpy arrays for images and masks
        """
        image_vol = []
        mask_vol = []
        final_num = min(self.start_num + self.batch_size, self.total_num)
        for num in range(start_num, final_num):
            img_name = f"volume_{num % self.total_num:03d}.nii.gz"  # Wrap around using modulus
            mask_name = f"volume_{num % self.total_num}.nii.gz"  # Wrap around using modulus
            img_path = os.path.join(self.image_dir, img_name)
            mask_path = os.path.join(self.masks_dir, mask_name)
            
            if os.path.exists(img_path) and os.path.exists(mask_path):
                nii_img = nib.load(img_path)
                image = nii_img.get_fdata().astype(np.float32)
                nii_mask = nib.load(mask_path)
                mask = nii_mask.get_fdata().astype(np.float32)
                depth_padding = 160 - image.shape[2]  # Assuming the shape is (160, 160, 155) and depth is 155
                if depth_padding > 0:
                    image = np.pad(image, ((0, 0), (0, 0), (0, depth_padding)), mode='constant', constant_values=0)
                    mask = np.pad(mask, ((0, 0), (0, 0), (0, depth_padding)), mode='constant', constant_values=0)
                
                image_vol.append(image)
                mask_vol.append(mask)
                
            else:
                print(f"File not found: {img_path} or {mask_path}")
        
        # Convert lists to numpy arrays and stack them
        return np.stack(image_vol, axis=0), np.stack(mask_vol, axis=0)

    def next(self):
        img, mask = self._load_and_stack_volumes(self.start_num)
        self.start_num = (self.start_num + self.batch_size)% self.total_num
        img, mask = tf.convert_to_tensor(img, dtype = tf.float32), tf.convert_to_tensor(mask, dtype = tf.float32)
        # return tf.expand_dims(img, axis = -1), tf.expand_dims(mask, axis = -1)
        return img, mask

In [27]:
image_dir = "data/Brats_resize/images"
mask_dir = "data/Brats_resize/masks"
batch_size = 4
total_num = 369
Dataloader = VolumeDataset(image_dir, mask_dir, batch_size, total_num)


In [28]:
img, mask = Dataloader.next()

In [29]:

print(img.shape)


(4, 160, 160, 160)


In [30]:
img.shape

TensorShape([4, 160, 160, 160])

In [31]:
def custom_train_loop(model, dataset, batch_size, total_num, epochs=10):
    # Setup the optimizer and loss function
    optimizer = Adam(learning_rate=1e-4)
    
    # For loss tracking and plotting
    loss_history = []
    steps_per_epoch = total_num//batch_size 

    # Training loop
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        epoch_loss = 0
        
        # Create a tqdm progress bar to visualize training
        progress_bar = tqdm(range(steps_per_epoch), desc="Training step", ncols=100)
        
        # Loop over dataset for steps_per_epoch steps
        for step in progress_bar:
            # Load a batch of data (images and masks)
            images, masks = dataset.next()

            # Start the gradient tape for training
            with tf.GradientTape() as tape:
                # Forward pass
                predictions = model(images)
                
                # Compute the loss
                loss = combined_loss(masks, predictions)  # Replace with your loss function

            # Compute gradients
            grads = tape.gradient(loss, model.trainable_variables)
            
            # Apply gradients to the model using the optimizer
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            
            # Update loss history
            epoch_loss += loss.numpy()

            # Update progress bar with loss
            progress_bar.set_postfix(loss=loss.numpy())
        
        # Average loss for the epoch
        avg_loss = epoch_loss / steps_per_epoch
        loss_history.append(avg_loss)
        print(f"Epoch {epoch + 1} Loss: {avg_loss:.4f}")

        # Optionally: save model checkpoint or intermediate results after each epoch

    return loss_history

In [34]:
from tensorflow.keras import layers, models

In [ ]:
import tensorflow as tf
from tensorflow import keras

def generate_all_intermediate_models(model):
    """
    Generate intermediate models for each layer of the original model.
    
    Args:
        model (keras.Model): The original model to extract intermediate outputs from
    
    Returns:
        list: A list of tuples containing (layer_name, intermediate_model)
    """
    # Ensure we have the input tensor
    inputs = model.inputs[0]
    
    # List to store all intermediate models
    intermediate_models = []
    
    # Iterate through layers (excluding the last output layer)
    for i, layer in enumerate(model.layers[:-1]):
        try:
            # Create an intermediate model up to this layer
            intermediate_model = keras.Model(
                inputs=inputs, 
                outputs=layer.output,
                name=f'intermediate_model_to_{layer.name}'
            )
            
            # Add to our list with layer name and the model
            intermediate_models.append((layer.name, intermediate_model))
        
        except Exception as e:
            print(f"Could not create intermediate model for layer {layer.name}: {e}")
    
    return intermediate_models

def print_intermediate_shapes(model, input_data):
    """
    Print output shapes for each intermediate model.
    
    Args:
        model (keras.Model): The original model
        input_data (numpy.ndarray): Input data to pass through the models
    """
    # Generate all intermediate models
    intermediate_models = generate_all_intermediate_models(model)
    
    # Print shapes for each intermediate model
    print("Intermediate Layer Output Shapes:")
    print("-" * 40)
    
    for layer_name, inter_model in intermediate_models:
        try:
            # Predict and get output shape
            output = inter_model.predict(input_data)
            print(f"Layer: {layer_name}")
            print(f"Output Shape: {output.shape}")
            print("-" * 40)
        except Exception as e:
            print(f"Error processing layer {layer_name}: {e}")

# Usage example:
# intermediate_models = generate_all_intermediate_models(model)

print_intermediate_shapes(model, img)

In [89]:
model.summary()

Model: "unet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
_______________________________________________________________________________________________

In [91]:
model.layers[0].input
model.get_layer("unet_conv_downarm_0_0" ).output

<tf.Tensor 'unet_conv_downarm_0_0/Elu:0' shape=(None, None, None, None, 24) dtype=float32>

In [77]:
intermediate_model = models.Model(inputs=model.get_input_at(0), outputs=model.layers[0].output)

AttributeError: 'tuple' object has no attribute 'layer'

In [68]:
import tensorflow as tf
from tensorflow import keras

def generate_unet_intermediate_models(model):
    """
    Generate intermediate models for each layer of a U-Net architecture.
    
    Args:
        model (keras.Model): The original U-Net model
    
    Returns:
        dict: A dictionary of intermediate models with layer names as keys
    """
    # Dictionary to store intermediate models
    intermediate_models = {}
    
    # Input tensor
    inputs = model.get_input_at(0)
    
    # Iterate through layers and create intermediate models
    for layer in model.layers:
        try:
            # Find the output of the current layer
            layer_output = layer.output
            
            # Create an intermediate model
            try:
                intermediate_model = keras.Model(
                    inputs=inputs, 
                    outputs=layer_output,
                    name=f'intermediate_model_{layer.name}'
                )
                
                # Store the intermediate model
                intermediate_models[layer.name] = intermediate_model
            
            except Exception as e:
                print(f"Error creating model for layer {layer.name}: {e}")
        
        except Exception as e:
            print(f"Error processing layer {layer.name}: {e}")
    
    return intermediate_models

def print_layer_output_shapes(model, input_data):
    """
    Print output shapes for each layer in the model.
    
    Args:
        model (keras.Model): The original model
        input_data (numpy.ndarray): Input data to pass through the model
    """
    # Create intermediate models
    intermediate_models = generate_unet_intermediate_models(model)
    
    # Print shapes for each intermediate model
    print("Layer Output Shapes:")
    print("-" * 40)
    
    for layer_name, inter_model in intermediate_models.items():
        try:
            # Predict and get output shape
            output = inter_model.predict(input_data)
            print(f"Layer: {layer_name}")
            print(f"Output Shape: {output.shape}")
            print("-" * 40)
        except Exception as e:
            print(f"Error processing layer {layer_name}: {e}")

# Usage example:
# intermediate_models = generate_unet_intermediate_models(model)
print_layer_output_shapes(model, img)

Error creating model for layer unet_input: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_conv_downarm_0_0: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_conv_downarm_0_1: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_bn_down_0: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_maxpool_0: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_conv_downarm_1_0: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_conv_downarm_1_1: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_bn_down_1: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_maxpool_1: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_conv_downarm_2_0: 'tuple' object has no attribute 'layer'
Error creating model for layer unet_conv_downarm_2_1: 'tuple' object has no attribute 'layer'
Error

In [61]:
inter_model = create_intermediate_model(model)

In [64]:
inter_model.summary()

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
____________________________________________________________________________________________

In [32]:
l = custom_train_loop(model, Dataloader, batch_size, total_num)

Epoch 1/10


Training step:   0%|                                                         | 0/92 [00:00<?, ?it/s]
/home/rajnish/miniconda3/envs/synthseg/lib/python3.8/site-packages/keras/backend/tensorflow_backend.py:3201: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if training is 1 or training is True:
/home/rajnish/miniconda3/envs/synthseg/lib/python3.8/site-packages/keras/backend/tensorflow_backend.py:3207: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif training is 0 or training is False:
/home/rajnish/miniconda3/envs/synthseg/lib/python3.8/site-packages/keras/backend/tensorflow_backend.py:3201: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if training is 1 or training is True:
/home/rajnish/miniconda3/envs/synthseg/lib/python3.8/site-packages/keras/backend/tensorflow_backend.py:3207: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif training is 0 or training is False:
/home/rajnish/miniconda3/envs/synthseg/lib/python3.8/site-packages/keras/back

ValueError: strides should be of length 1, 2 or 4 but was 3